# Question 13 - 238. Product of Array Except Self

Given an integer array `nums`, return an array `answer` such that `answer[i]` is equal to the product of all the elements of `nums` except `nums[i]`.

The product of any prefix or suffix of `nums` is **guaranteed** to fit in a **32-bit** integer.

You must write an algorithm that runs in `O(n)` time and **without using the division operation**.

**Example 1:**

    Input: nums = [1,2,3,4]
    Output: [24,12,8,6]

**Example 2:**

    Input: nums = [-1,1,0,-3,3]
    Output: [0,0,9,0,0]

**Constraints:**

- `2 <= nums.length <= 10^5`
- `-30 <= nums[i] <= 30`
- The product of any prefix or suffix of `nums` is **guaranteed** to fit in a **32-bit** integer.

**Follow up:** Can you solve the problem in `O(1)` extra space complexity? (The output array does **not** count as extra space for space complexity analysis.)

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Prefix and suffix products (two passes, reusing the output array)

📘 **Revise first:** [Pattern D · prefix & suffix](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (multiply all others) | O(n²) | O(1) extra |
| 1½ Prefix array + suffix array | O(n) | O(n) extra |
| 2️⃣ Optimized (prefix in output + running suffix) | O(n) | O(1) extra |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

For every position, multiply **all the other numbers** together (everything except the one at that position).

```
nums   = [1,  2,  3,  4]
answer = [24, 12, 8,  6]
          ↑ 2·3·4   ↑ 1·2·4 ...
```

**The obvious shortcut is banned:** "multiply everything, then divide by `nums[i]`". Division isn't allowed, and it would break on zeros anyway (you can't divide by 0).

**Everyday picture: a row of people, each blindfolded in turn.** Person `i` wants the product of everyone else. Split "everyone else" into two groups:
- **everyone to my left**
- **everyone to my right**

So **answer[i] = (product of the left side) × (product of the right side)**.

```
for index 2 (value 3):   [1, 2]  3  [4]
                          left       right
                          1·2 = 2  ×  4  = 8 ✅
```

And "product of everything to my left" grows one step at a time as you walk left to right. It's just the previous left-product times the previous number. Same for the right side, walking right to left.

### Step 0 · Clarify before coding

🗣️ *"For each index I need the product of all the other elements, without using division and in O(n) time. Zeros and negatives are allowed, and the products fit in 32-bit integers."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Division allowed? | **No.** | Rules out total ÷ nums[i]. It's also unsafe with zeros. |
| Zeros / negatives? | **Yes.** | Our method must handle them naturally. |
| Does the output array count as extra space? | **No.** | That's what makes the O(1)-extra-space follow-up possible. |
| Size? | Up to 10⁵. | O(n²) = 10¹⁰, too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** for each position, loop over every other position and multiply.

🗣️ *"The direct approach is a nested loop: for each index, multiply every element except that one. It's O(n²), about 10¹⁰ operations at n = 10⁵, so it's too slow."*

**The waste:** the answers for neighbouring positions share almost all of their factors. `answer[1] = 1·3·4` and `answer[2] = 1·2·4` both include `1` and `4`, yet we recompute everything from scratch every time.

In [1]:
class SolutionBrute:
    def productExceptSelf(self, nums: list[int]) -> list[int]:
        n = len(nums)
        answer = []
        for i in range(n):
            p = 1
            for j in range(n):
                if j != i:
                    p *= nums[j]
            answer.append(p)
        return answer

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** `answer[i] = (everything left of i) × (everything right of i)`, and both of those can be **built up incrementally**, one multiplication per step.

**Version A: the clear one.** Build two helper arrays:
- `left[i]` = product of `nums[0 … i−1]` (left of i). Start with `left[0] = 1` (nothing on the left, and 1 is "neutral" for multiplication), then `left[i] = left[i−1] × nums[i−1]`.
- `right[i]` = product of `nums[i+1 … n−1]`, built the same way from the right.
- `answer[i] = left[i] × right[i]`.

That's O(n) time but O(n) **extra** memory for the two helper arrays.

**Version B: the follow-up (O(1) extra).**
1. **Left pass:** store the left-products **directly in `answer`**.
2. **Right pass:** walk from the right, carrying a **single number** `right_product`, and multiply it into `answer[i]` as you go.

🗣️ *"Each answer is the product of everything to its left times everything to its right. I can compute all the left products in one pass and all the right products in a second pass. To avoid extra arrays, I store the left products directly in the output array, then sweep from the right with one running variable for the right product and multiply it in. No division, zeros just work, O(n) time and O(1) extra space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Nested loops | ❌ | O(n²). Recomputes shared factors. |
| Total product ÷ nums[i] | ❌ | **Not allowed**, and it breaks on zeros (needs special cases for one zero vs. two zeros). |
| Left array + right array | ✅ clearest | O(n), easy to explain and prove, but two extra arrays. A great first version to say out loud. |
| **Left products in the output + one running right product** | ✅ **best** | Same O(n) time, and O(1) extra memory: the output array does double duty. |
| Logarithms: exp(sum of logs) | ❌ | Floating-point errors, and it can't handle zeros or negatives. |

**Why zeros "just work":** we never divide. A zero simply makes every left or right product that includes it equal to 0, which is the correct answer.

In [2]:
class SolutionPrefixSuffix:
    def productExceptSelf(self, nums: list[int]) -> list[int]:
        n = len(nums)
        left = [1] * n          # left[i]  = product of nums[0..i-1]
        right = [1] * n         # right[i] = product of nums[i+1..n-1]
        for i in range(1, n):
            left[i] = left[i - 1] * nums[i - 1]
        for i in range(n - 2, -1, -1):
            right[i] = right[i + 1] * nums[i + 1]
        return [left[i] * right[i] for i in range(n)]


class Solution:
    def productExceptSelf(self, nums: list[int]) -> list[int]:
        n = len(nums)
        answer = [1] * n
        # pass 1 (left to right): answer[i] = product of everything LEFT of i
        for i in range(1, n):
            answer[i] = answer[i - 1] * nums[i - 1]
        # pass 2 (right to left): multiply in the product of everything RIGHT of i
        right_product = 1
        for i in range(n - 1, -1, -1):
            answer[i] *= right_product
            right_product *= nums[i]        # include nums[i] for the next (left) position
        return answer

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [1, 2, 3, 4]`

**Pass 1: left products** (stored in `answer`)

| i | answer[i] = answer[i−1] × nums[i−1] | answer |
|---|---|---|
| 0 | 1 (nothing to the left) | [1, 1, 1, 1] |
| 1 | 1 × 1 = 1 | [1, 1, 1, 1] |
| 2 | 1 × 2 = 2 | [1, 1, 2, 1] |
| 3 | 2 × 3 = 6 | [1, 1, 2, 6] |

**Pass 2: multiply in the right products**

| i | right_product (everything right of i) | answer[i] × right_product | right_product after × nums[i] |
|---|---|---|---|
| 3 | 1 | 6 × 1 = **6** | 4 |
| 2 | 4 | 2 × 4 = **8** | 12 |
| 1 | 12 | 1 × 12 = **12** | 24 |
| 0 | 24 | 1 × 24 = **24** | 24 |

Result `[24, 12, 8, 6]` ✅

**Narrated:** "First I walk left to right, writing down for each spot the product of everything before it: 1, 1, 2, 6. Then I walk back from the right, carrying the product of everything after the current spot. At the last spot nothing is on the right, so I multiply by 1. Moving left, the right-side product becomes 4, then 12, then 24, and I multiply each into the spot."

**Order matters in pass 2:** multiply `answer[i]` by `right_product` **before** adding `nums[i]` into `right_product`. Otherwise you'd include the number itself.

In [3]:
cases = [
    ([1, 2, 3, 4], [24, 12, 8, 6]),
    ([-1, 1, 0, -3, 3], [0, 0, 9, 0, 0]),
    ([0, 0], [0, 0]),           # two zeros -> everything is 0
    ([0, 5], [5, 0]),           # one zero
    ([2, 3], [3, 2]),
    ([-2, -3, 4], [-12, -8, 6]),
]
for nums, expected in cases:
    for Impl in (SolutionBrute, SolutionPrefixSuffix, Solution):
        assert Impl().productExceptSelf(nums) == expected, Impl.__name__
    print(nums, "->", expected)

import random
for _ in range(500):
    nums = [random.randint(-3, 3) for _ in range(random.randint(2, 8))]
    assert Solution().productExceptSelf(nums) == SolutionBrute().productExceptSelf(nums)
print("All tests passed ✅")

[1, 2, 3, 4] -> [24, 12, 8, 6]
[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0]
[0, 0] -> [0, 0]
[0, 5] -> [5, 0]
[2, 3] -> [3, 2]
[-2, -3, 4] -> [-12, -8, 6]
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Nested loops | O(n²) | O(1) | For every spot, multiply every other number. |
| Left + right arrays | O(n) | O(n) | Two passes, two helper arrays. |
| **Left in output + running right** | **O(n)** | **O(1)** | Two passes, one helper number. |

**Why O(n) time?** Two separate passes over the array, each doing one or two multiplications per spot. 2n steps is still "proportional to n".

**Why O(1) extra memory?** Apart from the required output array (which doesn't count), we keep a single number, `right_product`.

**Edge cases to mention:** one zero (only that position is non-zero), two or more zeros (everything is zero), negatives (signs work out automatically), n = 2.

**Likely follow-ups:**
- *If division were allowed?* → Count zeros. 0 zeros: `total // nums[i]`. Exactly 1 zero: only the zero's position gets the product of the others. 2+ zeros: all 0. More cases to get wrong, which is why prefix/suffix is preferred.
- *Where else does left/right precomputation show up?* → Candy (Q15), Trapping Rain Water (Q16), range-sum queries.

---

#### 🗣️ Full interview script (about 60 seconds)

*"For each index I need the product of all the other elements, without division and in linear time.*

*Brute force multiplies all the others for each index, which is O(n²). Division would be O(n), but it's not allowed and it breaks on zeros.*

*Instead I note that each answer is the product of everything to its left times everything to its right. I do a left-to-right pass storing the running left product into the output array. Then a right-to-left pass with a single variable holding the running right product: I multiply it into the output at each index, then fold the current number into it.*

*That's two passes, O(n) time, and O(1) extra space beyond the output. Zeros and negatives are handled naturally because I never divide."*